# Prepping Data Challenge - Week 15

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
adv_stock = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_15_adv_stock_per_store.csv')
adv_recall = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_15_adv_recall_items.csv')

In [3]:
adv_stock.head()

,City,Store,Store ID,Category,Product ID,Unit Price,Quantity,Date
0,Birmingahm,Birmingham,8,Groceries,GR1711,35.01,32,21/05/2025
1,London,Richmond,3,Groceries,GR5380,5.13,31,28/05/2025
2,Manchester,Chorlton,7,Groceries,GR9966,39.92,20,30/05/2025
3,London,Stratford,4,Groceries,GR1840,41.32,42,13/05/2025
4,London,Stratford,4,Groceries,GR5052,31.65,44,31/05/2025


In [4]:
adv_recall.head()

,Category,Product ID,Unit Price
0,Groceries,GR1840,41.32
1,Groceries,GR2261,38.20
2,Groceries,GR7679,40.06
3,Groceries,GR0294,11.77
4,Groceries,GR1705,47.16


Compare `Product ID` to check if there are more products in store, than products removed

In [5]:
adv_stock['Product ID'].nunique()

1241

In [6]:
adv_recall['Product ID'].nunique()

58

## Challenges

### Merge the datasets

In [7]:
adv_recall['Recalled'] = True

In [8]:
recall_data = adv_stock.merge(adv_recall, on=['Category', 'Product ID', 'Unit Price'], how='left')

### Find the days since recall announcement

In [9]:
RECALL_ANNOUNCEMENT = np.datetime64('2025-05-13')

In [10]:
recall_data['Date'] = pd.to_datetime(recall_data['Date'], format='%d/%m/%Y')

In [11]:
recall_data['Days since Announcement'] = np.where(recall_data['Recalled'] == True, (recall_data['Date'] - RECALL_ANNOUNCEMENT).dt.days, np.nan)

### Check if recall was on Target

In [12]:
recall_data['Recall Target'] = np.where(~recall_data['Days since Announcement'].isna(), np.where(recall_data['Days since Announcement'] <= 7, 'On Target', 'Overdue'), np.nan)

In [13]:
recall_data['Recall Target'] = np.where((recall_data['Date'].isna()) & (recall_data['Recalled'] == True), 'Incomplete', recall_data['Recall Target'])

In [14]:
recall_data['Recall Target'].unique()

array(['nan', 'On Target', 'Overdue', 'Incomplete'], dtype=object)

### Sort stores by removal speed

In [15]:
recall_speed_by_store = recall_data.groupby('Store')['Days since Announcement'].mean().sort_values().reset_index()

### Create days and hours columns

In [16]:
recall_speed_by_store['Days'] = recall_speed_by_store['Days since Announcement'].apply(lambda x: int(x))
recall_speed_by_store['Hours'] = recall_speed_by_store['Days since Announcement'].apply(lambda x: int((x - int(x)) * 24))
recall_speed_by_store = recall_speed_by_store[['Store', 'Days', 'Hours']].set_index('Store')

In [17]:
recall_speed_by_store

,Days,Hours
Store,,
Stratford,6,14
Richmond,7,12
Wembley,7,20
Nottingham,8,8
Trafford,8,15
Salford,8,17
Leeds,9,14
Birmingham,10,10
Chorlton,10,17


In [18]:
recall_speed_by_store.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/15_recall_speed_by_store.csv')

### Calculate overdue days

In [19]:
overdue_recalls = recall_data[recall_data['Recall Target'].isin(['Overdue', 'Incomplete'])].copy()

In [20]:
TODAYS_DATE = np.datetime64('2026-04-09')
MAX_OVERDUE = (TODAYS_DATE - RECALL_ANNOUNCEMENT) / np.timedelta64(1, "D")

In [21]:
overdue_recalls['Days since Announcement'] = np.where(overdue_recalls['Recall Target'] == "Incomplete", MAX_OVERDUE, overdue_recalls['Days since Announcement'])

In [22]:
overdue_recalls['Days Overdue'] = overdue_recalls['Days since Announcement'] - 7

In [23]:
overdue_information = overdue_recalls.groupby('Store')['Days Overdue'].mean().reset_index()

In [24]:
overdue_information['Quantity'] = overdue_recalls.groupby('Store')['Quantity'].sum().reset_index()['Quantity']

In [25]:
overdue_information

,Store,Days Overdue,Quantity
0,Birmingham,4.000000,126
1,Chorlton,5.400000,174
2,Leeds,4.444444,307
3,Nottingham,4.000000,148
4,Oxford Street,7.000000,161
5,Richmond,163.000000,51
6,Salford,5.250000,113
7,Stratford,4.500000,89
8,Trafford,49.571429,151
9,Wembley,8.666667,90


In [26]:
overdue_information.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/15_overdue_information.csv')